# Continuous batching: admit queued work at iteration boundaries

Prerequisites: autoregressive decode iterations and per-request KV state. Estimated time: 30 minutes. Objectives: trace queued admission at fixed capacity, compare slot utilization to static batching, and identify scheduler constraints.

A deterministic scheduler trace shows how a completed request releases a slot for a waiting request while another request remains active. This is scheduling arithmetic, not a GPU throughput benchmark.

## Workload and fixed-capacity baseline

Capacity is two concurrent requests. Requests A, B, and C arrive together with decode lengths 3, 1, and 2. Static batching runs A and B until both finish, then starts C. Count useful request-iterations and allocated slot-iterations.

In [ ]:
lengths = {'A': 3, 'B': 1, 'C': 2}
capacity = 2
static_groups = [['A', 'B'], ['C']]
static_trace = []
time = 0
for group in static_groups:
    duration = max(lengths[name] for name in group)
    for step in range(duration):
        active = [name for name in group if step < lengths[name]]
        static_trace.append((time, active, capacity - len(active)))
        time += 1
static_elapsed = time
static_useful = sum(lengths.values())
static_allocated = static_elapsed * capacity
assert static_elapsed == 5 and static_useful == 6 and static_allocated == 10
print('static trace (tick, active, idle slots):', static_trace)

## Continuous admission trace

At every decode boundary, retire completed requests, then admit the oldest queued request while capacity is available. Here B finishes after tick 0, so C is admitted at tick 1 while A is still active. In a real server, KV capacity and compatibility constraints also gate admission.

In [ ]:
remaining = dict(lengths)
queued = ['A', 'B', 'C']
active = []
continuous_trace = []
admitted_at = {}
tick = 0
while queued or active:
    while queued and len(active) < capacity:
        name = queued.pop(0)
        active.append(name)
        admitted_at[name] = tick
    current = active[:]
    continuous_trace.append((tick, current[:]))
    for name in current:
        remaining[name] -= 1
    active = [name for name in current if remaining[name] > 0]
    tick += 1
continuous_elapsed = tick
assert admitted_at == {'A': 0, 'B': 0, 'C': 1}
assert continuous_trace == [(0, ['A','B']), (1, ['A','C']), (2, ['A','C'])]
print('continuous trace:', continuous_trace, 'admissions:', admitted_at)

## Comparison and metric

The continuous schedule completes six useful request-iterations in three ticks using six allocated slot-iterations (100% toy slot utilization). Static groups take five ticks and allocate ten slot-iterations (60%). These are deliberately simplified slot metrics; they omit prefill, varied per-step costs, queueing arrivals, KV memory, and scheduler overhead.

In [ ]:
continuous_allocated = continuous_elapsed * capacity
static_utilization = static_useful / static_allocated
continuous_utilization = static_useful / continuous_allocated
assert continuous_elapsed == 3 and continuous_allocated == 6
assert static_utilization == 0.6 and continuous_utilization == 1.0
assert continuous_elapsed < static_elapsed
print('elapsed ticks static/continuous:', static_elapsed, continuous_elapsed, 'utilization:', static_utilization, continuous_utilization)

## Edge and failure cases

An empty queue should not create work. Admission must never exceed capacity. A long request can be delayed by policy even when short jobs keep arriving; production schedulers need wait-age or fairness controls and KV-memory limits.

In [ ]:
assert all(len(batch) <= capacity for _, batch in continuous_trace)
assert continuous_trace[-1][0] == continuous_elapsed - 1
empty_queue = []
assert not empty_queue  # no phantom admission
# A request requiring more KV tokens than the budget must be rejected/deferred.
kv_budget, request_kv_tokens = 8, 9
assert request_kv_tokens > kv_budget

## Exercises

1. Change B to four decode steps and compare static and continuous traces.
2. Add a per-request KV-token budget and defer admission when the active set would exceed it.
3. Add a maximum wait age or tenant quota and discuss its throughput/fairness trade-off.

## Summary

Continuous batching retires and admits requests at iteration boundaries. The trace demonstrates actual queued admission at fixed capacity and compares equivalent slot-iteration measures. Production evaluation must add TTFT, inter-token latency, KV occupancy, throughput, and fairness. See the [concept note](../concepts/26-continuous-batching.md) and the [practice assessment](../interview-prep/curriculum-practice.md#continuous-batching).